# Chapter 23: Metrics that Answer Different Questions

Ranking, coverage, and stopping measures.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
if str(ROOT / "examples") not in sys.path:
    sys.path.insert(0, str(ROOT / "examples"))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:
from retrieval_judgements import ranked_metrics, bundle_metrics, aggregate
import math

# Worked example from the chapter: B, U, A with grades A=2, B=1, U unjudged.
result = ranked_metrics(["B", "U", "A"], {"A": 2, "B": 1, "N": 0}, k=3)
print(result)
assert abs(result["precision@k"] - 2/3) < 1e-12
assert result["precision_upper@k"] == 1.0
assert result["recall@k"] == 1.0
assert result["rr@k"] == 1.0
assert abs(result["ap@k"] - 5/6) < 1e-12
expected_ndcg = 2.5 / (3 + 1 / math.log2(3))
assert abs(result["ndcg@k"] - expected_ndcg) < 1e-12
print("worked nDCG:", result["ndcg@k"])

In [ ]:
# Precision falls while recall rises as the prefix grows.
grades = {"a": 1, "b": 1, "c": 0}
one = ranked_metrics(["a"], grades, 1)
three = ranked_metrics(["a", "c", "b"], grades, 3)
print("P@1", one["precision@k"], "recall@1", one["recall@k"])
print("P@3", three["precision@k"], "recall@3", three["recall@k"])
assert three["recall@k"] > one["recall@k"]
assert three["precision@k"] < one["precision@k"]

In [ ]:
# Undefined metrics are not zeros when there are no known positives.
result = ranked_metrics(["a"], {"a": 0})
print(result)
assert all(result[k] is None for k in ("recall@k", "rr@k", "ap@k", "ndcg@k"))
# Empty prefix with known positives has zero recovery, not undefined recall.
empty = ranked_metrics([], {"a": 2})
print(empty)
assert empty["recall@k"] == 0
assert empty["ndcg@k"] == 0
assert empty["judged_fraction_returned"] is None

In [ ]:
# Unjudged returns are unknown, not certified negatives.
result = ranked_metrics(["x"], {})
print("unjudged returned:", result["unjudged_returned"], "precision upper:", result["precision_upper@k"])
assert result["unjudged_returned"] == 1
assert result["precision_upper@k"] == 0.25
assert result["recall@k"] is None

In [ ]:
# Bundle coverage needs all anchors; one unit is not always enough.
from document_identity import Collection, PUBLIC, REFERENCE_FIELDS
# Reuse manual units from setup by re-running minimal construction is verbose;
# instead exercise bundle_metrics with two distinct unit references.
import copy
u1 = {"source_id": "s", "document_id": "d", "version_id": "v", "representation_id": "r",
      "content_sha256": "h", "span": [0, 10], "passage_id": "p1"}
u2 = {"source_id": "s", "document_id": "d", "version_id": "v", "representation_id": "r",
      "content_sha256": "h", "span": [10, 20], "passage_id": "p2"}
anchor1 = {"source_id": "s", "document_id": "d", "version_id": "v", "representation_id": "r",
           "content_sha256": "h", "span": [0, 10]}
anchor2 = {"source_id": "s", "document_id": "d", "version_id": "v", "representation_id": "r",
           "content_sha256": "h", "span": [10, 20]}
bundle = [anchor1, anchor2]
metrics = bundle_metrics([bundle], [u1, u2])
print(metrics)
assert metrics["bundle_covered"] is True
assert metrics["single_unit_sufficient"] is False
assert bundle_metrics([], [u1, u2])["bundle_covered"] is None

In [ ]:
# Aggregate preserves undefined denominators.
agg = aggregate([{"ndcg": None}, {"ndcg": 0.5}], ["ndcg"])
print(agg)
assert agg["ndcg"] == {"mean": 0.5, "defined_queries": 1}

In [ ]:
# Duplicate results and invalid cutoffs/grades are rejected.
for bad in (["a", "a"],):
    try:
        ranked_metrics(bad, {"a": 1})
        raised = False
    except ValueError:
        raised = True
    assert raised
for cutoff in (0, True, -1):
    try:
        ranked_metrics([], {}, cutoff)
        raised = False
    except ValueError:
        raised = True
    assert raised
try:
    ranked_metrics(["a"], {"a": -1})
    raised = False
except ValueError:
    raised = True
assert raised
print("metric validation passed")

## Observed versus not established
Observed: declared metric formulas and edge cases. Not established: that any metric proves sufficiency, truth, or task success.